In [1]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.4-mini")

In [3]:
user_message_string = "Tell me a joke"

sys_message_string = "You tell jokes."

output_message_object = model.invoke([
    {"role": "system", "content": sys_message_string},
    {"role": "user", "content": user_message_string},
])

output_message_string = output_message_object.content

print(output_message_string)

Why don’t skeletons fight each other?

Because they don’t have the guts.


The "Langchain way of doing things"

In [4]:
from langchain.messages import HumanMessage, SystemMessage

user_message_string = "Tell me a joke"

sys_message_string = "You tell jokes."

output_message_object = model.invoke([
    SystemMessage(content=sys_message_string),
    HumanMessage(content=user_message_string),
])

output_message_string = output_message_object.content

print(output_message_string)

Why don’t skeletons fight each other?

Because they don’t have the guts.


## What is an Agent?

In [ ]:
from langchain.agents import create_agent

# Minimal agent — 3 lines
agent = create_agent(model="openai:gpt-5.4-mini", tools=[])
result = agent.invoke({"messages": "What is LangChain?"})
result["messages"][-1].pretty_print()

How can I get a simple LLM to use a tool?

In [20]:
from pydantic.functional_serializers import model_serializer


def parse_square_brackets_content(raw_text_output: str) -> str:
    """
    Parse the content inside square brackets from a given text.

    Args:
        raw_text_output (str): The input text containing square brackets.

    Returns:
        str: The content inside the square brackets.
    """
    # Find the start and end of the content inside square brackets
    start = raw_text_output.find("[")
    end = raw_text_output.find("]")
    if start == -1 or end == -1:
        return ""
    return raw_text_output[start + 1:end]

def parse_calculator_function_call(raw_text_output: str) -> str:
    """
    Parse the content inside square brackets from a given text.

    Args:
        raw_text_output (str): The input text containing square brackets.

    Returns:
        str: The content inside the square brackets.
    """
    # Find the start and end of the content inside square brackets
    start = raw_text_output.find("Calculator(")
    end = raw_text_output.find(") →")
    if start == -1 or end == -1:
        return ""
    return raw_text_output[start + 11:end]
    

def calculator_function(a: int, b: int, operation: str) -> int:
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

example_output = "Out of 1400 participants, 400 (or [Calculator(400 / 1400) → 0.29] 29%) passed the test."
user_msg_str = parse_square_brackets_content(example_output)

sys_msg_str = """
You are a helpful assistant that can calculate the percentage of participants who passed the test, consider
that you have access to a calculator function like this:

Input: Calculator(a, b, operation) → result

Output: calculator_function(a, b, operation) (don't include the arrow stuff)

If the user asks for a calculation you should not try to do it, but use the calculator function.
"""
model = init_chat_model("gpt-5.4")
output = model.invoke([
    SystemMessage(content=sys_msg_str),
    HumanMessage(content=user_msg_str),
])
print(output.content)

calculator_function(400, 1400, "/")


In [21]:
output_as_function = eval(output.content)
output_as_function

0.2857142857142857

Why use Langchain or Langgraph?